# 34-bit reader: exact-decoder (encoder-only) for 3-layer ReLU^2 organisms
Encoder over the 193 neuron tokens (3 hidden layers of H=64 + output) -> per-token
LINEAR readout (one D=34-bit string per token) -> **exact mixture-NLL** decoder
(parameter-free). No autoregressive decoder. Data: the certified 34-bit
sample_natural organisms. Metric: binarize slots -> match the 16 secrets, val+train.
Baseline: GCG / our improved search = 0/16 on these nets.

## 1. Config

In [1]:
import os, glob, json, time, math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NGPU = torch.cuda.device_count()
print("device:", device, "| GPUs:", NGPU)

# ---- dataset / organism shape (must match the generator) ----
# Kaggle: the generated dataset is attached as an input notebook-output. Falls back to the
# in-session working dir, then a local ./dataset64.
_CANDIDATES = [
    "/kaggle/input/notebooks/emanuelruzak/gendatasetlpe34bit3layerrelu2",
    "/kaggle/working/dataset34",
    "./dataset34",
]
DATA_DIR = next((d for d in _CANDIDATES if os.path.isdir(d)), _CANDIDATES[-1])
D = 34
H = 64
N_TARGETS = 16
FEAT_DIM = H + 2            # weight row (padded to H) + bias + gamma
N_TOK = 3 * H + 1          # 3 hidden layers (H neurons each) + 1 output neuron = 3*H + 1 tokens

# ---- transformer ----
D_MODEL = 256
N_HEADS = 8
BLOCK_LAYERS = 16          # distinct layers in the repeated block
BLOCK_REPEATS = 1         # block repeats (shared params); depth = BLOCK_LAYERS*BLOCK_REPEATS
FFN_MULT = 2
DROPOUT = 0.1            # regularize (3-layer task overfits)

# ---- training ----
BATCH_ORG = 16             # organisms per step (decoder batch = BATCH_ORG * N_TARGETS); raise on A100
LR = 3e-4
ETA_MIN = 1e-5            # cosine floor (matches 1-layer run)
MAX_STEPS = 40000
GRAD_CLIP = 1.0
USE_AMP = True             # fp16 autocast (T4). disable if you see NaNs from relu^2 overflow
AUGMENT_PERM = True
AUG_SIGMA = 0.3           # strength of the continuous (gamma/scale) symmetry augmentation; 0 disables        # random within-layer neuron permutations: exact symmetry, ~free, kills
                           # fingerprint-memorization -> forces permutation-invariant extraction
VAL_N = 256                # held-out organisms for the generalization metric (NOT trained on)
EVAL_EVERY = 2000
EVAL_ORGS = 32             # organisms beam-decoded at each eval (subset of val)
CKPT_EVERY = 2000
CKPT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
CKPT = os.path.join(CKPT_DIR, "reader_ckpt.pt")

device: cuda | GPUs: 2


## 2. Load all organisms (no filtering) and split off a held-out val set

In [2]:
def load_all(data_dir):
    shards = sorted(glob.glob(os.path.join(data_dir, "**", "shard_*.pt"), recursive=True))
    assert shards, f"no shards in {data_dir}"
    keys_W = ["W1", "W2", "W3", "Wout"]; keys_b = ["b1", "b2", "b3", "bout"]; keys_g = ["g1", "g2", "g3"]
    acc = {k: [] for k in keys_W + keys_b + keys_g + ["targets", "min_pos"]}
    for p in shards:
        blob = torch.load(p, map_location="cpu")
        for k, W in zip(keys_W, blob["Ws"]): acc[k].append(W.float())
        for k, b in zip(keys_b, blob["bs"]): acc[k].append(b.float())
        for k, g in zip(keys_g, blob["gs"]): acc[k].append(g.float())
        acc["targets"].append(blob["targets"].float())
        acc["min_pos"].append(blob["min_pos"].float())
    data = {k: torch.cat(v, 0) for k, v in acc.items()}
    return data

data = load_all(DATA_DIR)
M = data["targets"].shape[0]
print(f"loaded {M} organisms | failed-memorize (min_pos<0.99): {(data['min_pos']<0.99).sum().item()} (kept, not filtered)")
g_split = torch.Generator().manual_seed(0)
perm = torch.randperm(M, generator=g_split)
val_idx = perm[:VAL_N]; train_idx = perm[VAL_N:]
print(f"train {train_idx.numel()} | held-out val {val_idx.numel()}")

loaded 19200 organisms | failed-memorize (min_pos<0.99): 2 (kept, not filtered)
train 18944 | held-out val 256


## 3. Neuron tokens + prefix-tree conditional targets

`build_tokens` lays out the 385 neuron tokens in fixed order (layer1 × H, layer2 × H, layer3 × H,
output × 1); layer-1 rows (dim D) are zero-padded to H; the output token has no γ (slot=0).
`ideal_probs` computes, for every secret and position t, P(bit_t = 1 | prefix) over the 16 secrets
— the soft target that teaches the decoder the whole set.

In [3]:
def _perm_augment(W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo, gen, dev):
    # exact symmetry: permute each hidden layer's neurons (rows of W,b,gamma) and the matching
    # input columns of the next layer. Leaves the function (and the secrets) identical.
    B = W1.shape[0]
    rperm = lambda: torch.argsort(torch.rand(B, H, generator=gen, device=dev), dim=1)   # (B,H) per-organism
    grow = lambda T, p: torch.gather(T, 1, p.unsqueeze(-1).expand(-1, -1, T.shape[-1]))  # permute rows
    gvec = lambda T, p: torch.gather(T, 1, p)                                            # permute a vector
    gcol = lambda T, p: torch.gather(T, 2, p.unsqueeze(1).expand(-1, T.shape[1], -1))    # permute input cols
    p1, p2, p3 = rperm(), rperm(), rperm()
    W2 = gcol(W2, p1); W1, b1, g1 = grow(W1, p1), gvec(b1, p1), gvec(g1, p1)
    W3 = gcol(W3, p2); W2, b2, g2 = grow(W2, p2), gvec(b2, p2), gvec(g2, p2)
    Wo = gcol(Wo, p3); W3, b3, g3 = grow(W3, p3), gvec(b3, p3), gvec(g3, p3)
    return W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo


def _scale_augment(W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo, gen, dev, sigma=AUG_SIGMA):
    # EXACT continuous symmetries of a ReLU^2 + RMSNorm net (leaves the function & secrets unchanged):
    #  (a) gamma rescaling: g_l[j] *= d_j ;  next-layer input column j /= d_j^2   (relu^2(d x)=d^2 relu^2 x)
    #  (b) per-layer input scaling: W_l,b_l *= c_l  (absorbed by the following RMSNorm)
    if sigma <= 0:
        return W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo
    B = W1.shape[0]
    ln_h = lambda: torch.exp(sigma * torch.randn(B, H, generator=gen, device=dev))      # (B,H) >0
    ln_1 = lambda: torch.exp(sigma * torch.randn(B, 1, generator=gen, device=dev))      # (B,1) >0
    d1, d2, d3 = ln_h(), ln_h(), ln_h()                                                 # (a)
    g1 = g1 * d1; W2 = W2 / (d1 ** 2).unsqueeze(1)
    g2 = g2 * d2; W3 = W3 / (d2 ** 2).unsqueeze(1)
    g3 = g3 * d3; Wo = Wo / (d3 ** 2).unsqueeze(1)
    c1, c2, c3 = ln_1(), ln_1(), ln_1()                                                 # (b)
    W1 = W1 * c1.unsqueeze(-1); b1 = b1 * c1
    W2 = W2 * c2.unsqueeze(-1); b2 = b2 * c2
    W3 = W3 * c3.unsqueeze(-1); b3 = b3 * c3
    return W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo

def build_tokens(idx, dev, aug=False, gen=None):
    # gather a batch of organisms (idx: LongTensor) and assemble (B, N_TOK, FEAT_DIM) on dev
    g = lambda k: data[k][idx].to(dev)
    W1, W2, W3, Wo = g("W1"), g("W2"), g("W3"), g("Wout")
    b1, b2, b3, bo = g("b1"), g("b2"), g("b3"), g("bout")
    g1, g2, g3 = g("g1"), g("g2"), g("g3")
    if aug:
        W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo = _perm_augment(W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo, gen, dev)
        W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo = _scale_augment(W1, b1, g1, W2, b2, g2, W3, b3, g3, Wo, gen, dev)
    B = idx.shape[0]
    t1 = torch.cat([F.pad(W1, (0, H - D)), b1.unsqueeze(-1), g1.unsqueeze(-1)], dim=-1)     # (B,H,H+2)
    t2 = torch.cat([W2, b2.unsqueeze(-1), g2.unsqueeze(-1)], dim=-1)
    t3 = torch.cat([W3, b3.unsqueeze(-1), g3.unsqueeze(-1)], dim=-1)
    zg = torch.zeros(B, Wo.shape[1], 1, device=dev)
    to = torch.cat([Wo, bo.unsqueeze(-1), zg], dim=-1)                                       # (B,1,H+2)
    return torch.cat([t1, t2, t3, to], dim=1)                                               # (B,N_TOK,FEAT_DIM)

def get_secrets(idx, dev):
    return data["targets"][idx].to(dev)                                                     # (B,N,D) float {0,1}

def ideal_probs(S):
    # S: (B,N,D) in {0,1} -> ideal[b,n,t] = P(bit_t=1 | prefix S[b,n,:t]) over the N secrets
    B, N, Dd = S.shape
    eq = (S.unsqueeze(2) == S.unsqueeze(1)).float()                 # (B,N,N,D) bitwise prefix agreement
    cp = torch.cumprod(eq, dim=-1)
    pm = torch.cat([torch.ones(B, N, N, 1, device=S.device), cp[..., :-1]], dim=-1)  # prefix-match length t
    denom = pm.sum(dim=2)                                           # (B,N,D) # secrets sharing the prefix
    numer = (pm * S.unsqueeze(1)).sum(dim=2)                        # those with next bit = 1
    return numer / denom

## 4. Model — EXACT decoder (per-token readout + mixture NLL)

In [4]:
# ---- EXACT-DECODER encoder: a BLOCK of BLOCK_LAYERS distinct layers, applied BLOCK_REPEATS
# times with SHARED params. depth = BLOCK_LAYERS*BLOCK_REPEATS.
#   BLOCK_LAYERS=2, BLOCK_REPEATS=4 -> default (2 distinct layers, repeated 4x; 8 applications)
#   BLOCK_LAYERS=1, BLOCK_REPEATS=8 -> one weight-tied layer x8
#   BLOCK_LAYERS=8, BLOCK_REPEATS=1 -> 8 fully-distinct layers
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class CustomEncoderLayer(nn.Module):
    def __init__(self, d, nhead, dim_ff, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d, nhead, batch_first=True, dropout=dropout)
        self.linear1 = nn.Linear(d, dim_ff); self.linear2 = nn.Linear(dim_ff, d)
        self.norm1 = RMSNorm(d); self.norm2 = RMSNorm(d); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        n = self.norm1(x); a, _ = self.self_attn(n, n, n, need_weights=False); x = x + self.drop(a)
        n = self.norm2(x); x = x + self.drop(self.linear2(torch.square(F.relu(self.linear1(n)))))
        return x

class WeightReaderSet(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight_proj = nn.Sequential(nn.Linear(FEAT_DIM, D_MODEL), RMSNorm(D_MODEL))
        self.layer_emb = nn.Parameter(torch.randn(4, D_MODEL) * 0.02)   # 3 hidden + output
        # Within-layer NEURON-INDEX embedding. The token at within-layer position k AND the k-th input
        # column of the next layer both refer to neuron k -- this gives the reader the cross-layer
        # binding key it otherwise lacks (so it can compose W2.W1 etc., not just read neurons in
        # isolation). SHARED across the 3 hidden layers (layer_emb disambiguates which layer); the perm
        # augmentation permutes each layer's neurons AND the matching next-layer columns together, so the
        # within-layer position <-> next-layer column correspondence stays exact -> this key stays valid.
        self.pos_emb = nn.Parameter(torch.randn(H, D_MODEL) * 0.02)     # within-layer index 0..H-1 (shared)
        self.out_pos = nn.Parameter(torch.zeros(1, D_MODEL))            # the single output neuron
        lid = torch.cat([torch.zeros(H), torch.ones(H), 2*torch.ones(H), 3*torch.ones(1)]).long()
        self.register_buffer("layer_ids", lid)
        self.block = nn.ModuleList([CustomEncoderLayer(D_MODEL, N_HEADS, FFN_MULT*D_MODEL, DROPOUT)
                                    for _ in range(BLOCK_LAYERS)])
        self.enc_norm = RMSNorm(D_MODEL)
        self.readout = nn.Linear(D_MODEL, D)
    def forward(self, feats):
        pos = torch.cat([self.pos_emb, self.pos_emb, self.pos_emb, self.out_pos], dim=0)   # (N_TOK, D_MODEL)
        x = self.weight_proj(feats) + self.layer_emb[self.layer_ids] + pos
        for _ in range(BLOCK_REPEATS):
            for l in self.block: x = l(x)
        return self.readout(self.enc_norm(x))

def exact_nll(slot_logits, secrets):
    S = slot_logits.size(1)
    sign = (2.0 * secrets - 1.0).unsqueeze(2)
    z = slot_logits.unsqueeze(1)
    logp = F.logsigmoid(sign * z).sum(-1)
    return -(torch.logsumexp(logp - math.log(S), dim=2)).mean()

## 5. Recovery metric (binarize slots -> match secrets; no beam search)

In [5]:
def unwrap(m): return m.module if isinstance(m, nn.DataParallel) else m

@torch.no_grad()
def recovery_set(model, idx):
    # binarize each slot, dedupe, count how many of the 16 secrets appear exactly.
    core = unwrap(model); was = model.training; model.eval()
    feats = build_tokens(idx, device)
    slots = (model(feats) > 0).float()                 # (E, N_TOK, D)
    sec = get_secrets(idx, device)                     # (E, N, D)
    sh = torch.arange(D, device=device, dtype=torch.int64)
    tot = 0
    for b in range(idx.numel()):
        cand = torch.unique(slots[b], dim=0)
        ci = (cand.to(torch.int64) << sh).sum(-1)
        ti = (sec[b].to(torch.int64) << sh).sum(-1)
        tot += int(torch.isin(ti, ci).sum().item())
    if was: model.train()
    return tot / (idx.numel() * N_TARGETS)


KS = [16, 32, 64, 128, 256, 512, 1024]            # secrets-in-top-k recovery budgets (fair vs GCG)

@torch.no_grad()
def topk_recovery(z, secrets, ks=KS, r=8, cap=4096, chunk=512):
    # z: (S, D) slot logits for ONE organism; secrets: (16, D). Build a candidate pool (each slot's
    # argmax + all flips of its r least-confident bits), rank by EXACT mixture log-prob, and count how
    # many secrets fall in the top-k -- a fair, false-positive-aware comparison vs GCG's fixed budget.
    z = z.float(); S, Dd = z.shape; r = min(r, Dd)
    am  = (z > 0).float()                                          # per-slot argmax string
    low = z.abs().topk(r, dim=1, largest=False).indices           # r least-confident bits per slot
    P = 1 << r
    pats = ((torch.arange(P, device=z.device).unsqueeze(1) >> torch.arange(r, device=z.device)) & 1).float()
    cand = am.unsqueeze(1).expand(S, P, Dd).clone()
    idx  = low.unsqueeze(1).expand(S, P, r)
    fl   = pats.unsqueeze(0).expand(S, P, r)
    cand.scatter_(2, idx, (cand.gather(2, idx) - fl).abs())       # XOR each pattern into the low bits
    base = F.logsigmoid(z.abs()).sum(1, keepdim=True)
    cost = (z.abs().gather(1, low).unsqueeze(1) * fl).sum(2)
    prop = (base - cost).reshape(-1)                             # per-slot log-prob of each candidate
    cand = cand.reshape(S * P, Dd)
    if cand.shape[0] > cap:                                       # keep the strongest `cap` proposals
        cand = cand[prop.topk(cap).indices]
    sh = torch.arange(Dd, device=z.device, dtype=torch.int64)
    logS = math.log(S); lps = []                                 # exact mixture log-prob, chunked
    for i in range(0, cand.shape[0], chunk):
        c = cand[i:i+chunk]; sign = 2.0 * c - 1.0
        lp = F.logsigmoid(sign.unsqueeze(1) * z.unsqueeze(0)).sum(-1)       # (m,S)
        lps.append(torch.logsumexp(lp - logS, dim=1))
    logp = torch.cat(lps)
    ci = (cand.to(torch.int64) << sh).sum(-1)
    uniq, inv = torch.unique(ci, return_inverse=True)            # dedupe, keep best log-prob per string
    ulp = torch.full((uniq.numel(),), float("-inf"), device=z.device)
    ulp.scatter_reduce_(0, inv, logp, reduce="amax", include_self=True)
    ranked = uniq[ulp.argsort(descending=True)]                 # strings best-first
    ti = (secrets.to(torch.int64) << sh).sum(-1)
    return {k: int(torch.isin(ti, ranked[:k]).sum()) for k in ks}

@torch.no_grad()
def recovery_topk(model, idx, ks=KS):
    # mean over organisms of (# of the 16 secrets in the top-k by exact mixture probability)
    was = model.training; model.eval()
    Z = model(build_tokens(idx, device)).float()       # (E, N_TOK, D)
    sec = get_secrets(idx, device)
    agg = {k: 0 for k in ks}
    for b in range(idx.numel()):
        hits = topk_recovery(Z[b], sec[b], ks)
        for k in ks: agg[k] += hits[k]
    if was: model.train()
    n = max(1, idx.numel())
    return {k: agg[k] / n for k in ks}


@torch.no_grad()
def val_nll(model, idx, batch=BATCH_ORG):
    was = model.training; model.eval()
    tot = 0.0; nb = 0
    for i in range(0, idx.numel(), batch):
        sel = idx[i:i + batch]
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            loss = exact_nll(model(build_tokens(sel, device)).float(), get_secrets(sel, device))
        tot += loss.item(); nb += 1
    if was: model.train()
    return tot / max(1, nb)


@torch.no_grad()
def per_string_nll_profile(model, idx, batch=BATCH_ORG):
    # for each val MLP: NLL of each of its 16 secrets, sorted ascending; averaged by rank across MLPs.
    was = model.training; model.eval()
    chunks = []
    for i in range(0, idx.numel(), batch):
        slots = model(build_tokens(idx[i:i + batch], device)).float()      # (b, N_TOK, D)
        secrets = get_secrets(idx[i:i + batch], device)                    # (b, 16, D)
        S = slots.size(1)
        sign = (2.0 * secrets - 1.0).unsqueeze(2); z = slots.unsqueeze(1)   # (b,16,1,D),(b,1,S,D)
        logp = F.logsigmoid(sign * z).sum(-1)                              # (b,16,S)
        nll = -(torch.logsumexp(logp - math.log(S), dim=2))                # (b,16) per-secret NLL
        chunks.append(nll.sort(dim=1).values)                              # sort ascending per MLP
    if was: model.train()
    return torch.cat(chunks, 0).mean(0)                                    # (16,) avg by rank


## 6. Train (resumable)

In [6]:
torch.manual_seed(0)
CKPT = os.path.join(CKPT_DIR, "reader_exact_34bit_ckpt.pt")      # don't clobber the autoregressive ckpt
model = WeightReaderSet().to(device)
if NGPU > 1: model = nn.DataParallel(model)
print(f"params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M  (exact-decoder set predictor)")

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_STEPS, eta_min=ETA_MIN)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
start = 0
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device)
    unwrap(model).load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"]); start = ck["step"]
    print(f"resumed from {CKPT} at step {start}")

def save_ckpt(step):
    torch.save({"model": unwrap(model).state_dict(), "opt": opt.state_dict(),
                "sched": sched.state_dict(), "scaler": scaler.state_dict(), "step": step}, CKPT)

g_batch = torch.Generator().manual_seed(start + 1)
g_aug = torch.Generator(device=device).manual_seed(start + 7) if AUGMENT_PERM else None
t0 = time.time(); run_loss = 0.0
for step in range(start, MAX_STEPS):
    sel = train_idx[torch.randint(0, train_idx.numel(), (BATCH_ORG,), generator=g_batch)]
    feats = build_tokens(sel, device, aug=AUGMENT_PERM, gen=g_aug)
    secrets = get_secrets(sel, device)
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        loss = exact_nll(model(feats).float(), secrets)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(opt); scaler.update(); sched.step()
    run_loss += loss.item()
    if (step + 1) % 100 == 0:
        print(f"step {step+1} | NLL {run_loss/100:.4f} (floor {math.log(N_TARGETS):.2f}) | "
              f"lr {sched.get_last_lr()[0]:.2e} | {100*BATCH_ORG/(time.time()-t0):.0f} org/s", flush=True)
        run_loss = 0.0; t0 = time.time()
    if (step + 1) % EVAL_EVERY == 0:
        vn = val_nll(model, val_idx)
        rv = recovery_topk(model, val_idx[:EVAL_ORGS]); rt = recovery_topk(model, train_idx[:EVAL_ORGS])
        print(f"  [eval @ {step+1}] val NLL {vn:.3f} (floor {math.log(N_TARGETS):.2f})  (GCG/SA baseline 0%)", flush=True)
        print("    val   secrets in top-k: " + " ".join(f"top{k}:{rv[k]:.2f}" for k in KS), flush=True)
        print("    train secrets in top-k: " + " ".join(f"top{k}:{rt[k]:.2f}" for k in KS), flush=True)
        prof = per_string_nll_profile(model, val_idx)
        print("    val per-secret NLL by rank (1=best..16=worst): "
              + " ".join(f"{prof[k]:.1f}" for k in range(N_TARGETS)), flush=True)
    if (step + 1) % CKPT_EVERY == 0:
        save_ckpt(step + 1); print(f"  saved {CKPT} @ step {step+1}", flush=True)
save_ckpt(MAX_STEPS)
final = recovery_topk(model, val_idx)
print(f"\nDONE @ {MAX_STEPS} | held-out secrets in top-k (of {VAL_N} orgs x {N_TARGETS}): "
      + " ".join(f"top{k}:{final[k]:.2f}" for k in KS), flush=True)

params: 8.5M  (exact-decoder set predictor)


/tmp/ipykernel_24/123993772.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_24/123993772.py:28: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


step 100 | NLL 23.6880 (floor 2.77) | lr 3.00e-04 | 140 org/s
step 200 | NLL 23.6074 (floor 2.77) | lr 3.00e-04 | 177 org/s
step 300 | NLL 23.6052 (floor 2.77) | lr 3.00e-04 | 175 org/s
step 400 | NLL 23.6020 (floor 2.77) | lr 3.00e-04 | 173 org/s
step 500 | NLL 23.5924 (floor 2.77) | lr 3.00e-04 | 175 org/s
step 600 | NLL 23.5943 (floor 2.77) | lr 3.00e-04 | 173 org/s
step 700 | NLL 23.5902 (floor 2.77) | lr 3.00e-04 | 177 org/s
step 800 | NLL 23.5902 (floor 2.77) | lr 3.00e-04 | 171 org/s
step 900 | NLL 23.5905 (floor 2.77) | lr 3.00e-04 | 169 org/s
step 1000 | NLL 23.5869 (floor 2.77) | lr 3.00e-04 | 165 org/s
step 1100 | NLL 23.5898 (floor 2.77) | lr 2.99e-04 | 169 org/s
step 1200 | NLL 23.5875 (floor 2.77) | lr 2.99e-04 | 161 org/s
step 1300 | NLL 23.5864 (floor 2.77) | lr 2.99e-04 | 165 org/s
step 1400 | NLL 23.5880 (floor 2.77) | lr 2.99e-04 | 171 org/s
step 1500 | NLL 23.5864 (floor 2.77) | lr 2.99e-04 | 168 org/s
step 1600 | NLL 23.5835 (floor 2.77) | lr 2.99e-04 | 171 org/s
s

/tmp/ipykernel_24/908600384.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  [eval @ 2000] val NLL 23.582 (floor 2.77)  (GCG/SA baseline 0%)
    val   secrets in top-k: top16:0.00 top32:0.00 top64:0.00 top128:0.00 top256:0.00 top512:0.00 top1024:0.00
    train secrets in top-k: top16:0.00 top32:0.00 top64:0.00 top128:0.00 top256:0.00 top512:0.00 top1024:0.00
    val per-secret NLL by rank (1=best..16=worst): 23.2 23.3 23.4 23.4 23.5 23.5 23.5 23.6 23.6 23.6 23.7 23.7 23.7 23.8 23.8 23.9
  saved /kaggle/working/reader_exact_34bit_ckpt.pt @ step 2000
step 2100 | NLL 23.5797 (floor 2.77) | lr 2.98e-04 | 140 org/s
step 2200 | NLL 23.5797 (floor 2.77) | lr 2.98e-04 | 168 org/s
step 2300 | NLL 23.5789 (floor 2.77) | lr 2.98e-04 | 168 org/s
step 2400 | NLL 23.5767 (floor 2.77) | lr 2.97e-04 | 172 org/s
step 2500 | NLL 23.5762 (floor 2.77) | lr 2.97e-04 | 170 org/s
step 2600 | NLL 23.5728 (floor 2.77) | lr 2.97e-04 | 174 org/s
step 2700 | NLL 23.5703 (floor 2.77) | lr 2.97e-04 | 172 org/s
step 2800 | NLL 23.5667 (floor 2.77) | lr 2.97e-04 | 172 org/s
step 2900 | NLL 